# Annotator Results Summary

This notebook summarises the human-annotator evaluation of the candidate clustering taxonomies.

Each annotator scored the **same 16 clustering trials** (drawn from 4 sources: `gpt5_nano`, `apertus`, `data_points`, `phi4_mini`).
For every trial they rated each topic on two axes:

- **cohesion** (0–100) — how internally consistent the topic is
- **fit** (1–5) — how well the topic fits the intended taxonomy

and the app stored per-trial aggregates (`cohesion_mean`, `fit_mean`, `overall`).

Files used per annotator folder:

| file | meaning |
|------|---------|
| `evaluation_submitted.json` | final submitted ratings (core data) |
| `evaluation_order.json` | order trials were shown in |
| `evaluation_timer.json` | time spent on the evaluation task |
| `moderation_timer.json` | time spent on the moderation task |
| `top_trials.json` | trial metadata (dbcv, ccc, cluster count) — identical across annotators |
| `moderated_taxonomy.json` / `placed_keys.json` | output of the (optional) moderation step |

In [ ]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Base directory = folder containing this notebook (with a sensible fallback).
try:
    BASE = Path(__file__).resolve().parent  # noqa: F821
except NameError:
    BASE = Path.cwd()
if not sorted(BASE.glob("annotator*/evaluation_submitted.json")):
    # fallback when launched from a parent dir
    cand = BASE / "annotator_results"
    if cand.exists():
        BASE = cand

ANNOTATOR_DIRS = sorted(p for p in BASE.glob("annotator*") if p.is_dir())
print("Base directory:", BASE)
print("Annotators found:", [p.name for p in ANNOTATOR_DIRS])

## 1. Load the data

We build a **tidy long table** — one row per (annotator, source, trial, topic) — plus a per-trial aggregate table from the values the app already computed.

In [ ]:
def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


topic_rows = []     # one row per rated topic
trial_rows = []     # one row per rated trial (app aggregates)
meta_rows = []      # annotator-level metadata (time, counts)

for adir in ANNOTATOR_DIRS:
    name = adir.name
    submitted = load_json(adir / "evaluation_submitted.json")
    ratings = submitted.get("ratings", {})

    for key, rec in ratings.items():
        source, trial = key.split(":")
        trial = int(trial)
        trial_rows.append({
            "annotator": name,
            "source": source,
            "trial": trial,
            "n_topics": len(rec.get("topics", {})),
            "cohesion_mean": rec.get("cohesion_mean"),
            "fit_mean": rec.get("fit_mean"),
            "overall": rec.get("overall"),
        })
        for topic_id, scores in rec.get("topics", {}).items():
            topic_rows.append({
                "annotator": name,
                "source": source,
                "trial": trial,
                "topic": int(topic_id),
                "cohesion": scores.get("cohesion"),
                "fit": scores.get("fit"),
            })

    # timers (seconds -> minutes)
    eval_t = load_json(adir / "evaluation_timer.json").get("elapsed_seconds") if (adir / "evaluation_timer.json").exists() else None
    mod_path = adir / "moderation_timer.json"
    mod_t = load_json(mod_path).get("elapsed_seconds") if mod_path.exists() else None
    has_moderation = (adir / "moderated_taxonomy.json").exists()
    meta_rows.append({
        "annotator": name,
        "submitted_at": submitted.get("submitted_at"),
        "n_trials_rated": len(ratings),
        "eval_minutes": (eval_t / 60) if eval_t else None,
        "moderation_minutes": (mod_t / 60) if mod_t else None,
        "did_moderation": has_moderation,
    })

topics_df = pd.DataFrame(topic_rows)
trials_df = pd.DataFrame(trial_rows)
meta_df = pd.DataFrame(meta_rows)

print(f"{len(topics_df):,} topic ratings | {len(trials_df):,} trial ratings | {len(meta_df)} annotators")
topics_df.head()

In [ ]:
# Attach trial metadata (dbcv / ccc / cluster count) from top_trials.json.
# This file is shared across annotators, so load it once from any annotator that has it.
trial_meta = {}
source_labels = {}
for adir in ANNOTATOR_DIRS:
    tt_path = adir / "top_trials.json"
    if tt_path.exists():
        for t in load_json(tt_path):
            trial_meta[(t["source"], t["trial"])] = {
                "source_label": t.get("source_label"),
                "dbcv": t.get("dbcv"),
                "ccc": t.get("ccc"),
                "metric_avg": t.get("avg"),
                "n_clusters": t.get("meta", {}).get("clusters"),
                "outlier_ratio": t.get("meta", {}).get("outlier_ratio"),
            }
            source_labels[t["source"]] = t.get("source_label")
        break

meta_lookup = pd.DataFrame([{"source": s, "trial": tr, **v} for (s, tr), v in trial_meta.items()])
trials_df = trials_df.merge(meta_lookup, on=["source", "trial"], how="left")
topics_df["source_label"] = topics_df["source"].map(source_labels)
trials_df["trial_id"] = trials_df["source"] + ":" + trials_df["trial"].astype(str)
trials_df.head()

## 2. Annotator overview

How many trials each annotator rated, time spent, and whether they completed the moderation step.

In [ ]:
meta_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(len(meta_df))
w = 0.4
ax.bar(x - w / 2, meta_df["eval_minutes"].fillna(0), w, label="evaluation")
ax.bar(x + w / 2, meta_df["moderation_minutes"].fillna(0), w, label="moderation")
ax.set_xticks(x)
ax.set_xticklabels(meta_df["annotator"])
ax.set_ylabel("minutes")
ax.set_title("Time spent per annotator")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Per-trial scores

The `overall` score (0–100) is the app's headline rating for each trial. Below we average it across annotators and rank the trials.

In [ ]:
trial_summary = (
    trials_df
    .groupby(["source", "trial"])
    .agg(
        source_label=("source_label", "first"),
        n_clusters=("n_clusters", "first"),
        dbcv=("dbcv", "first"),
        ccc=("ccc", "first"),
        overall_mean=("overall", "mean"),
        overall_std=("overall", "std"),
        cohesion_mean=("cohesion_mean", "mean"),
        fit_mean=("fit_mean", "mean"),
        n_annotators=("annotator", "nunique"),
    )
    .reset_index()
    .sort_values("overall_mean", ascending=False)
)
trial_summary

In [ ]:
# Ranked bar chart of mean overall score per trial, coloured by source.
ts = trial_summary.copy()
ts["label"] = ts["source"] + ":" + ts["trial"].astype(str)
sources = ts["source"].unique()
cmap = {s: c for s, c in zip(sources, plt.cm.tab10.colors)}

fig, ax = plt.subplots(figsize=(9, 6))
y = np.arange(len(ts))[::-1]
ax.barh(y, ts["overall_mean"], xerr=ts["overall_std"].fillna(0),
        color=[cmap[s] for s in ts["source"]], capsize=3)
ax.set_yticks(y)
ax.set_yticklabels(ts["label"])
ax.set_xlabel("mean overall score (0–100)")
ax.set_title("Mean overall score per trial (± std across annotators)")
handles = [plt.Rectangle((0, 0), 1, 1, color=cmap[s]) for s in sources]
ax.legend(handles, [source_labels.get(s, s) for s in sources], title="source", loc="lower right")
plt.tight_layout()
plt.show()

## 4. Per-source (model family) comparison

Aggregating trials by their source model family.

In [ ]:
source_summary = (
    trials_df
    .groupby("source")
    .agg(
        source_label=("source_label", "first"),
        n_trials=("trial", "nunique"),
        overall_mean=("overall", "mean"),
        cohesion_mean=("cohesion_mean", "mean"),
        fit_mean=("fit_mean", "mean"),
    )
    .reset_index()
    .sort_values("overall_mean", ascending=False)
)
source_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
order = source_summary["source"]
labels = source_summary["source_label"].fillna(source_summary["source"])

axes[0].bar(labels, source_summary["overall_mean"], color=[cmap[s] for s in order])
axes[0].set_title("Mean overall score by source")
axes[0].set_ylabel("overall (0–100)")
axes[0].tick_params(axis="x", rotation=30)

# distribution of per-topic scores by source
data = [topics_df.loc[topics_df["source"] == s, "cohesion"].dropna() for s in order]
axes[1].boxplot(data, labels=labels)
axes[1].set_title("Per-topic cohesion distribution by source")
axes[1].set_ylabel("cohesion (0–100)")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 5. Human scores vs. automatic clustering metrics

Do the annotators' `overall` scores agree with the automatic clustering quality metrics (DBCV, CCC) that were used to select these trials?

In [ ]:
corr_df = trial_summary.dropna(subset=["dbcv", "ccc", "overall_mean"])
if len(corr_df) >= 3:
    print("Pearson correlation with mean overall score:")
    print(f"  DBCV       : {corr_df['overall_mean'].corr(corr_df['dbcv']):+.3f}")
    print(f"  CCC        : {corr_df['overall_mean'].corr(corr_df['ccc']):+.3f}")
    print(f"  n_clusters : {corr_df['overall_mean'].corr(corr_df['n_clusters']):+.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, metric in zip(axes, ["dbcv", "ccc"]):
    for s in sources:
        sub = corr_df[corr_df["source"] == s]
        ax.scatter(sub[metric], sub["overall_mean"], color=cmap[s],
                   label=source_labels.get(s, s), s=60)
    ax.set_xlabel(metric.upper())
    ax.set_ylabel("mean overall (0–100)")
    ax.set_title(f"Human score vs {metric.upper()}")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Inter-annotator agreement

How consistently did the annotators score the same trials? We pivot the per-trial `overall` score so each column is an annotator, then look at pairwise correlation and per-trial spread.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

pivot = trials_df.pivot_table(index=["source", "trial"], columns="annotator", values="overall")
corr = pivot.corr()
print("Pairwise correlation of 'overall' score across annotators:")
display(corr)

spread = pivot.std(axis=1).rename("std_across_annotators")
print(f"
Mean per-trial std across annotators: {spread.mean():.1f} points")

# Sequential colormap anchored on the requested tone #CCD8E7 (a light blue-grey),
# ramping from white (weak agreement) through that tone to a darker shade (strong).
cmap = LinearSegmentedColormap.from_list(
    "ccd8e7", ["#ffffff", "#CCD8E7", "#7d97b8", "#3d566f"]
)

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(corr.values, cmap=cmap, vmin=0, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=30, ha="right")
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        v = corr.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if v > 0.6 else "#1a2733", fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pearson r")
ax.set_title("Inter-annotator agreement (overall score)")
plt.tight_layout()
plt.show()

# Trials with the largest disagreement, for reference.
pivot.assign(std=spread).sort_values("std", ascending=False).head(10)

In [ ]:
# Heatmap-style view of each annotator's overall score per trial.
p = pivot.copy()
p.index = [f"{s}:{t}" for s, t in p.index]
fig, ax = plt.subplots(figsize=(6, 8))
im = ax.imshow(p.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(p.columns)))
ax.set_xticklabels(p.columns, rotation=30, ha="right")
ax.set_yticks(range(len(p.index)))
ax.set_yticklabels(p.index)
for i in range(p.shape[0]):
    for j in range(p.shape[1]):
        v = p.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.0f}", ha="center", va="center",
                    color="white" if v < p.values[~np.isnan(p.values)].mean() else "black", fontsize=8)
fig.colorbar(im, ax=ax, label="overall score")
ax.set_title("Overall score per trial × annotator")
plt.tight_layout()
plt.show()

## 7. Moderation step

After scoring, annotators could (optionally) build a **moderated taxonomy** by placing data points into topics. Here we summarise how far each annotator got.

In [ ]:
def count_tree(node):
    """Recursively count topics and placed data points in a moderated-taxonomy tree."""
    n_topics = 0
    n_points = len(node.get("data_points", []) or [])
    children = node.get("children", []) or []
    for ch in children:
        if isinstance(ch, dict):
            n_topics += 1
            ct, cp = count_tree(ch)
            n_topics += ct
            n_points += cp
    return n_topics, n_points


mod_rows = []
for adir in ANNOTATOR_DIRS:
    mt_path = adir / "moderated_taxonomy.json"
    pk_path = adir / "placed_keys.json"
    if not mt_path.exists():
        mod_rows.append({"annotator": adir.name, "completed_moderation": False,
                         "n_topics": 0, "n_placed_points": 0, "n_placed_keys": 0})
        continue
    tree = load_json(mt_path)
    n_topics, n_points = count_tree(tree)
    n_keys = len(load_json(pk_path)) if pk_path.exists() else None
    mod_rows.append({
        "annotator": adir.name,
        "completed_moderation": True,
        "root_topic": tree.get("topic_name"),
        "n_topics": n_topics,
        "n_placed_points": n_points,
        "n_placed_keys": n_keys,
    })

pd.DataFrame(mod_rows)

## 8. Key takeaways

This cell prints a short, auto-generated summary of the headline numbers.

In [ ]:
print("=" * 60)
print("ANNOTATOR RESULTS — SUMMARY")
print("=" * 60)
print(f"Annotators            : {len(ANNOTATOR_DIRS)} ({', '.join(p.name for p in ANNOTATOR_DIRS)})")
print(f"Trials evaluated      : {trials_df['trial'].nunique()} (each rated by all annotators)")
print(f"Total topic ratings   : {len(topics_df):,}")
print(f"Mean eval time        : {meta_df['eval_minutes'].mean():.0f} min")
print()
best = trial_summary.iloc[0]
print(f"Top trial (overall)   : {best['source']}:{best['trial']} "
      f"({best['source_label']}) — {best['overall_mean']:.1f}/100")
print("\nBest source by mean overall:")
for _, r in source_summary.iterrows():
    print(f"  {str(r['source_label'] or r['source']):28s} {r['overall_mean']:5.1f}/100  "
          f"(cohesion {r['cohesion_mean']:.0f}, fit {r['fit_mean']:.1f})")
print(f"\nMean inter-annotator spread (overall): {pivot.std(axis=1).mean():.1f} points")